# 07 -- Final Visualizations

Generate publication-quality figures summarising the EigenDialectos analysis.
Uses the dedicated `visualization` module functions with the project colour palette.

**Key modules:** `visualization.spectral_plots`, `visualization.dialect_maps`,
`visualization.gradient_plots`, `visualization.tensor_plots`, `visualization.embedding_plots`

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

from eigendialectos.constants import DialectCode, DIALECT_NAMES
from eigendialectos.types import EmbeddingMatrix, EigenDecomposition, DialectalSpectrum
from eigendialectos.visualization._colors import DIALECT_COLORS, dialect_label

# Output directory for saved figures
FIG_DIR = Path("../outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Figure output directory: {FIG_DIR.resolve()}")

## Setup: Build Synthetic Data for All Plots

Compute embeddings, transformations, eigendecompositions, and spectra that
will feed into all visualizations below.

In [ ]:
from eigendialectos.spectral.transformation import compute_transformation_matrix, compute_all_transforms
from eigendialectos.spectral.eigendecomposition import eigendecompose
from eigendialectos.spectral.eigenspectrum import compute_eigenspectrum
from eigendialectos.spectral.entropy import compute_dialectal_entropy, compare_entropies
from eigendialectos.spectral.distance import compute_distance_matrix

rng = np.random.default_rng(42)
dim, vocab_size = 30, 100
vocab = [f"w{i}" for i in range(vocab_size)]

base = rng.standard_normal((dim, vocab_size)).astype(np.float64)
embeddings = {}
for code in DialectCode:
    if code == DialectCode.ES_PEN:
        data = base.copy()
    else:
        scale = 0.1 + 0.2 * rng.random()
        data = base + rng.standard_normal((dim, vocab_size)) * scale
    embeddings[code] = EmbeddingMatrix(data=data, vocab=vocab, dialect_code=code)

transforms = compute_all_transforms(embeddings, reference=DialectCode.ES_PEN)
eigendecomps = {}
spectra = {}
entropies = {}

for code, W in transforms.items():
    eigen = eigendecompose(W)
    spectrum = compute_eigenspectrum(eigen)
    eigendecomps[code] = eigen
    spectra[code] = spectrum
    entropies[code] = spectrum.entropy

print(f"Computed spectra for {len(spectra)} dialect varieties")

## Figure 1: Eigenvalue Bar Chart

Top-k eigenvalue magnitudes grouped by dialect variety.

In [ ]:
from eigendialectos.visualization.spectral_plots import plot_eigenvalue_bars

fig = plot_eigenvalue_bars(spectra, top_k=15, save_path=FIG_DIR / "fig1_eigenvalue_bars.png")
plt.show()
print("Saved: fig1_eigenvalue_bars.png")

## Figure 2: Eigenvalue Decay Curves

How eigenvalue magnitudes decay across the spectrum for each variety.

In [ ]:
from eigendialectos.visualization.spectral_plots import plot_eigenvalue_decay

fig = plot_eigenvalue_decay(spectra, save_path=FIG_DIR / "fig2_eigenvalue_decay.png")
plt.show()
print("Saved: fig2_eigenvalue_decay.png")

## Figure 3: Cumulative Spectral Energy

Cumulative energy as a function of the number of eigenvalues retained.

In [ ]:
from eigendialectos.visualization.spectral_plots import plot_cumulative_energy

fig = plot_cumulative_energy(spectra, save_path=FIG_DIR / "fig3_cumulative_energy.png")
plt.show()
print("Saved: fig3_cumulative_energy.png")

## Figure 4: Spectral Entropy Comparison

Bar chart of dialectal spectral entropy values with mean reference line.

In [ ]:
from eigendialectos.visualization.spectral_plots import plot_entropy_comparison

fig = plot_entropy_comparison(entropies, save_path=FIG_DIR / "fig4_entropy_comparison.png")
plt.show()
print("Saved: fig4_entropy_comparison.png")

## Figure 5: Eigenspectrum Heatmap

Full eigenvalue magnitudes as a heatmap (dialects x eigenvalue index).

In [ ]:
from eigendialectos.visualization.spectral_plots import plot_eigenspectrum_heatmap

fig = plot_eigenspectrum_heatmap(spectra, save_path=FIG_DIR / "fig5_eigenspectrum_heatmap.png")
plt.show()
print("Saved: fig5_eigenspectrum_heatmap.png")

## Figure 6: Dialect Distance Matrix

Annotated heatmap of pairwise Frobenius distances between dialect transformation matrices.

In [ ]:
from eigendialectos.visualization.dialect_maps import plot_dialect_distance_matrix

labels = sorted(transforms.keys(), key=lambda c: c.value)
n = len(labels)
dist = np.zeros((n, n))
for i, ci in enumerate(labels):
    for j, cj in enumerate(labels):
        if i < j:
            diff = transforms[ci].data - transforms[cj].data
            d = float(np.linalg.norm(diff, "fro"))
            dist[i, j] = d
            dist[j, i] = d

fig = plot_dialect_distance_matrix(dist, labels, save_path=FIG_DIR / "fig6_distance_matrix.png")
plt.show()
print("Saved: fig6_distance_matrix.png")

## Figure 7: Dialect Dendrogram

Hierarchical clustering (Ward's method) of dialect varieties based on spectral distance.

In [ ]:
from eigendialectos.visualization.dialect_maps import plot_dialect_dendrogram

fig = plot_dialect_dendrogram(dist, labels, save_path=FIG_DIR / "fig7_dendrogram.png")
plt.show()
print("Saved: fig7_dendrogram.png")

## Figure 8: MDS Dialect Map

2D multidimensional scaling projection of dialect varieties from the distance matrix.

In [ ]:
from eigendialectos.visualization.dialect_maps import plot_dialect_mds

fig = plot_dialect_mds(dist, labels, save_path=FIG_DIR / "fig8_mds_map.png")
plt.show()
print("Saved: fig8_mds_map.png")

## Figure 9: DIAL Alpha Gradient

Metrics as a function of dialectal intensity alpha for a selected dialect variety.

In [ ]:
from eigendialectos.visualization.gradient_plots import plot_alpha_gradient
from eigendialectos.generative.dial import apply_dial

target_code = DialectCode.ES_RIO
eigen = eigendecomps[target_code]
alphas = list(np.arange(0.0, 1.55, 0.1).round(2))

frob_norms = []
dist_from_I = []
cond_numbers = []
for alpha in alphas:
    W_a = apply_dial(eigen, alpha)
    frob_norms.append(float(np.linalg.norm(W_a.data, "fro")))
    dist_from_I.append(float(np.linalg.norm(W_a.data - np.eye(dim), "fro")))
    cond_numbers.append(float(np.linalg.cond(W_a.data)))

gradient_metrics = {
    "||W(a)||_F": frob_norms,
    "||W(a)-I||_F": dist_from_I,
    "cond(W(a))": cond_numbers,
}

fig = plot_alpha_gradient(
    alphas, gradient_metrics, target_code,
    save_path=FIG_DIR / "fig9_alpha_gradient.png"
)
plt.show()
print("Saved: fig9_alpha_gradient.png")

## Figure 10: Feature Activation Heatmap

How the eigenvalue spectrum changes across alpha values -- a heatmap of
eigenvalue magnitudes at each intensity level.

In [ ]:
from eigendialectos.visualization.gradient_plots import plot_feature_activation_heatmap

# Build feature activations: top-k eigenvalue magnitudes at each alpha
top_k = 10
features = {}
for k in range(top_k):
    feat_vals = []
    for alpha in alphas:
        W_a = apply_dial(eigen, alpha)
        ev = np.sort(np.abs(np.linalg.eigvals(W_a.data)))[::-1]
        feat_vals.append(float(ev[k]) if k < len(ev) else 0.0)
    features[f"lambda_{k}"] = feat_vals

fig = plot_feature_activation_heatmap(
    alphas, features,
    save_path=FIG_DIR / "fig10_feature_heatmap.png"
)
plt.show()
print("Saved: fig10_feature_heatmap.png")

## Figure 11: Tensor Factor Loadings

Mode-3 factor loadings from Tucker decomposition -- shows dialect groupings
in reduced tensor factor space.

In [ ]:
from eigendialectos.visualization.tensor_plots import plot_factor_loadings_heatmap
from eigendialectos.tensor.construction import build_dialect_tensor
from eigendialectos.tensor.tucker import tucker_decompose

tensor = build_dialect_tensor(transforms)
tucker_result = tucker_decompose(tensor, ranks=(10, 10, 4))
C = tucker_result["factor_matrices"][2]  # mode-3 factor: (n_dialects, r3)

fig = plot_factor_loadings_heatmap(
    C, tensor.dialect_codes,
    save_path=FIG_DIR / "fig11_factor_loadings.png"
)
plt.show()
print("Saved: fig11_factor_loadings.png")

## Figure 12: CP Components

Top-k CP decomposition components with weights and factor matrices.

In [ ]:
from eigendialectos.visualization.tensor_plots import plot_cp_components
from eigendialectos.tensor.cp import cp_decompose

cp_result = cp_decompose(tensor, rank=5, n_restarts=3)

fig = plot_cp_components(
    cp_result["weights"], cp_result["factor_matrices"], top_k=5,
    save_path=FIG_DIR / "fig12_cp_components.png"
)
plt.show()
print("Saved: fig12_cp_components.png")

## Figure 13: Reconstruction Scree Plot

CP reconstruction error as a function of tensor rank.

In [ ]:
from eigendialectos.visualization.tensor_plots import plot_reconstruction_scree

ranks = [1, 2, 3, 5, 8, 10]
errors = []
for r in ranks:
    try:
        res = cp_decompose(tensor, rank=r, n_restarts=2)
        errors.append(res["reconstruction_error"])
    except Exception:
        errors.append(np.nan)

fig = plot_reconstruction_scree(
    errors, ranks,
    save_path=FIG_DIR / "fig13_reconstruction_scree.png"
)
plt.show()
print("Saved: fig13_reconstruction_scree.png")

## Figure 14: Embedding PCA Variance

PCA explained variance for the embedding spaces.

In [ ]:
from eigendialectos.visualization.embedding_plots import plot_pca_variance

fig = plot_pca_variance(embeddings, save_path=FIG_DIR / "fig14_pca_variance.png")
plt.show()
print("Saved: fig14_pca_variance.png")

## Summary of Generated Figures

In [ ]:
fig_files = sorted(FIG_DIR.glob("*.png"))
print(f"Generated {len(fig_files)} publication figures in {FIG_DIR.resolve()}:\n")
for f in fig_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<40} {size_kb:>8.1f} KB")